<a href="https://colab.research.google.com/github/VivekGaddam/EDAVassignment/blob/main/EDAVAssignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# EDA & Visualization Assignment
### Dataset: Student Performance in Exams
**Source:** https://www.kaggle.com/datasets/spscientist/students-performance-in-exams

In [3]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('/content/drive/MyDrive/StudentsPerformance.csv')
df.head()

,gender,race/ethnicity,parental level of education,lunch,test preparation course,math score,reading score,writing score
0,female,group B,bachelor's degree,standard,none,72,72,74
1,female,group C,some college,standard,completed,69,90,88
2,female,group B,master's degree,standard,none,90,95,93
3,male,group A,associate's degree,free/reduced,none,47,57,44
4,male,group C,some college,standard,none,76,78,75


## Q1. Score Distributions & Variance Analysis
Load the dataset, create a **Series** for math, reading, and writing scores.  
Use `describe()` to examine distributions and identify the subject with the **highest variance**.

In [4]:
math_scores    = df['math score']
reading_scores = df['reading score']
writing_scores = df['writing score']

print('=' * 55)
print('           MATH SCORE — describe()')
print('=' * 55)
print(math_scores.describe())
print('\n' + '=' * 55)
print('        READING SCORE — describe()')
print('=' * 55)
print(reading_scores.describe())
print('\n' + '=' * 55)
print('        WRITING SCORE — describe()')
print('=' * 55)
print(writing_scores.describe())

           MATH SCORE — describe()
count    1000.00000
mean       66.08900
std        15.16308
min         0.00000
25%        57.00000
50%        66.00000
75%        77.00000
max       100.00000
Name: math score, dtype: float64

        READING SCORE — describe()
count    1000.000000
mean       69.169000
std        14.600192
min        17.000000
25%        59.000000
50%        70.000000
75%        79.000000
max       100.000000
Name: reading score, dtype: float64

        WRITING SCORE — describe()
count    1000.000000
mean       68.054000
std        15.195657
min        10.000000
25%        57.750000
50%        69.000000
75%        79.000000
max       100.000000
Name: writing score, dtype: float64


In [5]:
# Compute variances
variances = pd.Series({
    'math score':    math_scores.var(),
    'reading score': reading_scores.var(),
    'writing score': writing_scores.var()
})

print('Variance by Subject:')
print(variances)

highest_var_subject = variances.idxmax()
print(f'\nSubject with HIGHEST variance: {highest_var_subject} ({variances.max():.2f})')

Variance by Subject:
math score       229.918998
reading score    213.165605
writing score    230.907992
dtype: float64

Subject with HIGHEST variance: writing score (230.91)


## Q2. Effect of Test Preparation Course
Use `groupby()` on `test preparation course` with index-aligned operations to compare mean scores.  
Determine if preparation **significantly improves** performance across all three subjects.

In [6]:
# Group by test preparation course
prep_group = df.groupby('test preparation course')[['math score', 'reading score', 'writing score']]

prep_mean = prep_group.mean()
prep_std  = prep_group.std()

print('Mean Scores by Test Preparation Course:')
print(prep_mean.round(2))

print('\nStd Deviation:')
print(prep_std.round(2))

# Index-aligned difference
diff = prep_mean.loc['completed'] - prep_mean.loc['none']
print('\nScore Improvement (completed - none):')
print(diff.round(2))

# % improvement
pct_improvement = (diff / prep_mean.loc['none']) * 100
print('\n% Improvement:')
print(pct_improvement.round(2))

# Verdict
all_positive = all(diff > 0)
print(f'\nDoes preparation significantly improve ALL subjects? -> {"YES" if all_positive else "NO"}')

Mean Scores by Test Preparation Course:
                         math score  reading score  writing score
test preparation course                                          
completed                     69.70          73.89          74.42
none                          64.08          66.53          64.50

Std Deviation:
                         math score  reading score  writing score
test preparation course                                          
completed                     14.44          13.64          13.38
none                          15.19          14.46          15.00

Score Improvement (completed - none):
math score       5.62
reading score    7.36
writing score    9.91
dtype: float64

% Improvement:
math score        8.77
reading score    11.06
writing score    15.37
dtype: float64

Does preparation significantly improve ALL subjects? -> YES


## Q3. Hierarchical Indexing — Education & Gender
Apply **hierarchical indexing** using `parental level of education` and `gender`.  
Examine average scores using `xs()` and `loc[]`. Identify patterns between education level and performance.

In [7]:
# Build hierarchical index
df_hier = df.set_index(['parental level of education', 'gender'])
score_cols = ['math score', 'reading score', 'writing score']

# Average scores at each level
avg_scores = df_hier[score_cols].groupby(level=[0, 1]).mean().round(2)

print('Average Scores by Parental Education Level & Gender:')
print(avg_scores)

# Using xs() to slice a specific education level
edu_level = "master's degree"
print(f"\nxs() — Scores for '{edu_level}':")
print(avg_scores.xs(edu_level, level='parental level of education'))

# Using loc[]
print("\nloc[] — Scores for 'high school' > 'female':")
try:
    print(avg_scores.loc[('high school', 'female')])
except:
    print(avg_scores.xs('high school', level=0).loc['female'])

# Education-performance pattern
edu_avg = df.groupby('parental level of education')[score_cols].mean().mean(axis=1).sort_values(ascending=False)
print('\nAverage Overall Score by Education Level (sorted):')
print(edu_avg.round(2))

Average Scores by Parental Education Level & Gender:
                                    math score  reading score  writing score
parental level of education gender                                          
associate's degree          female       65.25          74.12          74.00
                            male         70.76          67.43          65.41
bachelor's degree           female       68.35          77.29          78.38
                            male         70.58          68.09          67.65
high school                 female       59.35          68.20          66.69
                            male         64.71          61.48          58.54
master's degree             female       66.50          76.81          77.64
                            male         74.83          73.13          72.61
some college                female       65.41          73.55          74.05
                            male         69.01          64.99          63.15
some high school       

## Q4. Academic Performance Index (API)
Create a **weighted composite API** (Math 40%, Reading 30%, Writing 30%) using ufunc operations.  
Rank all students and display a detailed attribute breakdown of the **top and bottom performers**.

In [8]:
# Weighted API: Math 40%, Reading 30%, Writing 30%
weights = np.array([0.40, 0.30, 0.30])
scores_matrix = df[['math score', 'reading score', 'writing score']].values

# ufunc-style weighted operation using np.dot
df['API'] = np.dot(scores_matrix, weights)

# Rank students (1 = best)
df['Rank'] = df['API'].rank(ascending=False, method='min').astype(int)

print('API Statistics:')
print(df['API'].describe().round(2))

# Top 10 performers
top10 = df.nlargest(10, 'API')[['gender','race/ethnicity','parental level of education',
                                 'test preparation course','math score','reading score',
                                 'writing score','API','Rank']]
print('\nTOP 10 Performers:')
print(top10.to_string(index=False))

# Bottom 10 performers
bot10 = df.nsmallest(10, 'API')[['gender','race/ethnicity','parental level of education',
                                   'test preparation course','math score','reading score',
                                   'writing score','API','Rank']]
print('\nBOTTOM 10 Performers:')
print(bot10.to_string(index=False))

API Statistics:
count    1000.00
mean       67.60
std        14.24
min         8.10
25%        58.10
50%        68.30
75%        77.60
max       100.00
Name: API, dtype: float64

TOP 10 Performers:
gender race/ethnicity parental level of education test preparation course  math score  reading score  writing score   API  Rank
female        group E           bachelor's degree                    none         100            100            100 100.0     1
  male        group E           bachelor's degree               completed         100            100            100 100.0     1
female        group E          associate's degree                    none         100            100            100 100.0     1
female        group E           bachelor's degree               completed          99            100            100  99.6     4
female        group D                some college                    none          98            100             99  98.9     5
female        group D            s

## Q5. Null Value Imputation Strategies
Introduce controlled nulls in **10% of rows**.  
Compare **mean, median, and mode** imputation strategies and assess their impact on group-level statistics.

In [9]:
np.random.seed(123)

# Work on a copy
df_null = df[['gender','test preparation course','math score','reading score','writing score']].copy()

# Introduce 10% nulls
n_rows = len(df_null)
null_idx = np.random.choice(n_rows, size=int(0.1 * n_rows), replace=False)

for col in ['math score', 'reading score', 'writing score']:
    chosen = np.random.choice(null_idx, size=len(null_idx)//2, replace=False)
    df_null.loc[chosen, col] = np.nan

print('Null counts after introduction:')
print(df_null.isnull().sum())
print(f'\nTotal nulls: {df_null.isnull().sum().sum()}')

score_cols_q5 = ['math score', 'reading score', 'writing score']

# Strategy 1: Mean imputation
df_mean = df_null.copy()
for col in score_cols_q5:
    df_mean[col] = df_mean[col].fillna(df_mean[col].mean())

# Strategy 2: Median imputation
df_median = df_null.copy()
for col in score_cols_q5:
    df_median[col] = df_median[col].fillna(df_median[col].median())

# Strategy 3: Mode imputation
df_mode = df_null.copy()
for col in score_cols_q5:
    df_mode[col] = df_mode[col].fillna(df_mode[col].mode()[0])

# Compare group-level statistics
print('\nGroup-Level Mean Comparison (by test preparation course):')
for label, dframe in [('With Nulls', df_null), ('Mean Imp', df_mean),
                       ('Median Imp', df_median), ('Mode Imp', df_mode)]:
    grp = dframe.groupby('test preparation course')[score_cols_q5].mean().round(2)
    print(f'\n--- {label} ---')
    print(grp)

# Overall stat comparison
comparison = pd.DataFrame({
    'original_mean':   df[score_cols_q5].mean(),
    'mean_imp_mean':   df_mean[score_cols_q5].mean(),
    'median_imp_mean': df_median[score_cols_q5].mean(),
    'mode_imp_mean':   df_mode[score_cols_q5].mean(),
    'original_std':    df[score_cols_q5].std(),
    'mean_imp_std':    df_mean[score_cols_q5].std(),
    'median_imp_std':  df_median[score_cols_q5].std(),
    'mode_imp_std':    df_mode[score_cols_q5].std(),
}).round(4)
print('\nOverall Imputation Strategy Comparison:')
print(comparison.T)

Null counts after introduction:
gender                      0
test preparation course     0
math score                 50
reading score              50
writing score              50
dtype: int64

Total nulls: 150

Group-Level Mean Comparison (by test preparation course):

--- With Nulls ---
                         math score  reading score  writing score
test preparation course                                          
completed                     69.52          73.72          74.41
none                          64.08          66.43          64.46

--- Mean Imp ---
                         math score  reading score  writing score
test preparation course                                          
completed                     69.38          73.48          74.18
none                          64.19          66.56          64.67

--- Median Imp ---
                         math score  reading score  writing score
test preparation course                                          
completed 

## Q6. Boolean Indexing — High Achievers (>90 in All Subjects)
Filter students who scored **above 90 in all three subjects** using boolean indexing.  
Examine their demographic profile using `value_counts()` and cross-tabulation.

In [10]:
# Boolean mask
mask = (df['math score'] > 90) & (df['reading score'] > 90) & (df['writing score'] > 90)
top_students = df[mask].copy()

print(f'Students scoring >90 in ALL subjects: {len(top_students)}')

print('\nGender Distribution:')
print(top_students['gender'].value_counts())

print('\nRace/Ethnicity Distribution:')
print(top_students['race/ethnicity'].value_counts())

print('\nParental Education Distribution:')
print(top_students['parental level of education'].value_counts())

print('\nTest Preparation Course:')
print(top_students['test preparation course'].value_counts())

print('\nCross-tabulation: Gender x Test Preparation:')
print(pd.crosstab(top_students['gender'], top_students['test preparation course']))

print('\nCross-tabulation: Race/Ethnicity x Test Preparation:')
print(pd.crosstab(top_students['race/ethnicity'], top_students['test preparation course']))

Students scoring >90 in ALL subjects: 23

Gender Distribution:
gender
female    17
male       6
Name: count, dtype: int64

Race/Ethnicity Distribution:
race/ethnicity
group E    9
group C    5
group D    5
group A    2
group B    2
Name: count, dtype: int64

Parental Education Distribution:
parental level of education
bachelor's degree     9
associate's degree    6
some college          4
some high school      2
master's degree       2
Name: count, dtype: int64

Test Preparation Course:
test preparation course
completed    14
none          9
Name: count, dtype: int64

Cross-tabulation: Gender x Test Preparation:
test preparation course  completed  none
gender                                  
female                          10     7
male                             4     2

Cross-tabulation: Race/Ethnicity x Test Preparation:
test preparation course  completed  none
race/ethnicity                          
group A                          1     1
group B                          1     

## Q7. Letter Grade Assignment using apply()
Design a **grading function** using `apply()` that assigns letter grades (A–F).  
Construct a full grade report DataFrame with **hierarchical indexing** on gender and race/ethnicity.

In [11]:
def assign_grade(score):
    if   score >= 90: return 'A'
    elif score >= 80: return 'B'
    elif score >= 70: return 'C'
    elif score >= 60: return 'D'
    else:             return 'F'

# Apply grading function to each score column
df['math grade']    = df['math score'].apply(assign_grade)
df['reading grade'] = df['reading score'].apply(assign_grade)
df['writing grade'] = df['writing score'].apply(assign_grade)

# Grade report with hierarchical index on gender + race/ethnicity
grade_report = df[['gender', 'race/ethnicity',
                    'math score', 'math grade',
                    'reading score', 'reading grade',
                    'writing score', 'writing grade']].copy()

grade_report = grade_report.set_index(['gender', 'race/ethnicity']).sort_index()

print('Grade Report (first 20 rows):')
print(grade_report.head(20))

print('\nMath Grade Distribution:')
print(df['math grade'].value_counts().sort_index())

print('\nReading Grade Distribution:')
print(df['reading grade'].value_counts().sort_index())

print('\nWriting Grade Distribution:')
print(df['writing grade'].value_counts().sort_index())

print('\nGrade Distribution by Gender:')
for subject_grade in ['math grade', 'reading grade', 'writing grade']:
    print(f'\n  {subject_grade}:')
    print(pd.crosstab(df['gender'], df[subject_grade]))

Grade Report (first 20 rows):
                       math score math grade  reading score reading grade  \
gender race/ethnicity                                                       
female group A                 50          F             53             F   
       group A                 55          F             65             D   
       group A                 41          F             51             F   
       group A                 58          F             70             C   
       group A                 51          F             49             F   
       group A                 44          F             64             D   
       group A                 71          C             83             B   
       group A                 38          F             43             F   
       group A                 49          F             65             D   
       group A                 59          F             85             B   
       group A                 47          F  

## Q8. Index Alignment — Student Score vs Class Average
Compute the **difference between each student's score and the class average** using index alignment.  
Identify students who **underperform** in specific subjects.

In [12]:
score_cols = ['math score', 'reading score', 'writing score']

# Class average
class_avg = df[score_cols].mean()
print('Class Average:')
print(class_avg.round(2))

# Index-aligned subtraction: student score - class average
deviation = df[score_cols].sub(class_avg)
print('\nDeviation from Class Average (first 10 rows):')
print(deviation.head(10).round(2))

print('\nDeviation Statistics:')
print(deviation.describe().round(2))

# Underperformers per subject
print(f'\nStudents below average in Math:    {(deviation["math score"] < 0).sum()}')
print(f'Students below average in Reading: {(deviation["reading score"] < 0).sum()}')
print(f'Students below average in Writing: {(deviation["writing score"] < 0).sum()}')

# Students below average in 2+ subjects
below_2plus = (deviation < 0).sum(axis=1) >= 2
print(f'\nStudents below average in 2+ subjects: {below_2plus.sum()} ({below_2plus.mean()*100:.1f}%)')

# Students below average in ALL three
all_below = (deviation < 0).all(axis=1)
print(f'Students below average in ALL 3 subjects: {all_below.sum()} ({all_below.mean()*100:.1f}%)')

Class Average:
math score       66.09
reading score    69.17
writing score    68.05
dtype: float64

Deviation from Class Average (first 10 rows):
   math score  reading score  writing score
0        5.91           2.83           5.95
1        2.91          20.83          19.95
2       23.91          25.83          24.95
3      -19.09         -12.17         -24.05
4        9.91           8.83           6.95
5        4.91          13.83           9.95
6       21.91          25.83          23.95
7      -26.09         -26.17         -29.05
8       -2.09          -5.17          -1.05
9      -28.09          -9.17         -18.05

Deviation Statistics:
       math score  reading score  writing score
count     1000.00        1000.00        1000.00
mean         0.00           0.00          -0.00
std         15.16          14.60          15.20
min        -66.09         -52.17         -58.05
25%         -9.09         -10.17         -10.30
50%         -0.09           0.83           0.95
75%        

## Q9. DataFrame Operations — Prep vs No-Prep
Split students into two DataFrames by preparation status.  
Use **subtraction and percentage operations** with aligned indices to quantify score differences between groups.

In [13]:
score_cols = ['math score', 'reading score', 'writing score']

# Split into two DataFrames
df_completed = df[df['test preparation course'] == 'completed'][score_cols].reset_index(drop=True)
df_none      = df[df['test preparation course'] == 'none'][score_cols].reset_index(drop=True)

print(f'Completed preparation: {len(df_completed)} students')
print(f'No preparation:        {len(df_none)} students')

# Group-level mean for aligned operations
mean_completed = df_completed.mean()
mean_none      = df_none.mean()

# Subtraction with index alignment
score_diff = mean_completed.sub(mean_none)
print('\nMean Score Difference (completed - none):')
print(score_diff.round(2))

# Percentage operation
pct_change = (score_diff / mean_none) * 100
print('\n% Change:')
print(pct_change.round(2))

# Std deviation comparison
std_diff = df_completed.std().sub(df_none.std())
print('\nStd Deviation Difference (completed - none):')
print(std_diff.round(2))

# Variance ratio
var_ratio = df_completed.var() / df_none.var()
print('\nVariance Ratio (completed / none):')
print(var_ratio.round(3))

Completed preparation: 358 students
No preparation:        642 students

Mean Score Difference (completed - none):
math score       5.62
reading score    7.36
writing score    9.91
dtype: float64

% Change:
math score        8.77
reading score    11.06
writing score    15.37
dtype: float64

Std Deviation Difference (completed - none):
math score      -0.75
reading score   -0.83
writing score   -1.62
dtype: float64

Variance Ratio (completed / none):
math score       0.904
reading score    0.889
writing score    0.795
dtype: float64


## Q10. Comprehensive Student Analytics Report
Build a full analytics report covering **null analysis, hierarchical grouping, ufunc transformations, and grade distribution**.

In [14]:
print('=' * 60)
print('      COMPREHENSIVE STUDENT ANALYTICS REPORT')
print('=' * 60)

# --- 1. NULL ANALYSIS ---
print('\n-- 1. NULL ANALYSIS --')
print(f'Dataset shape: {df.shape}')
print(f'Total null values: {df.isnull().sum().sum()}')
print(df.isnull().sum())

# --- 2. DATASET OVERVIEW ---
print('\n-- 2. DATASET OVERVIEW --')
print(df[['math score','reading score','writing score','API']].describe().round(2))

      COMPREHENSIVE STUDENT ANALYTICS REPORT

-- 1. NULL ANALYSIS --
Dataset shape: (1000, 13)
Total null values: 0
gender                         0
race/ethnicity                 0
parental level of education    0
lunch                          0
test preparation course        0
math score                     0
reading score                  0
writing score                  0
API                            0
Rank                           0
math grade                     0
reading grade                  0
writing grade                  0
dtype: int64

-- 2. DATASET OVERVIEW --
       math score  reading score  writing score      API
count     1000.00        1000.00        1000.00  1000.00
mean        66.09          69.17          68.05    67.60
std         15.16          14.60          15.20    14.24
min          0.00          17.00          10.00     8.10
25%         57.00          59.00          57.75    58.10
50%         66.00          70.00          69.00    68.30
75%         77.0

In [15]:
# --- 3. HIERARCHICAL GROUPING ---
print('-- 3. HIERARCHICAL GROUPING --')
hier_group = df.groupby(['gender', 'race/ethnicity'])[['math score','reading score','writing score']].mean().round(2)
print('Average Scores by Gender x Race/Ethnicity:')
print(hier_group)

edu_group = df.groupby(['parental level of education', 'gender'])[['math score','reading score','writing score']].mean().round(2)
print('\nAverage Scores by Parental Education x Gender:')
print(edu_group)

prep_group_report = df.groupby(['test preparation course', 'gender'])[['math score','reading score','writing score']].mean().round(2)
print('\nAverage Scores by Test Preparation x Gender:')
print(prep_group_report)

-- 3. HIERARCHICAL GROUPING --
Average Scores by Gender x Race/Ethnicity:
                       math score  reading score  writing score
gender race/ethnicity                                          
female group A              58.53          69.00          67.86
       group B              61.40          71.08          70.05
       group C              62.03          71.94          71.78
       group D              65.25          74.05          75.02
       group E              70.81          75.84          75.54
male   group A              63.74          61.74          59.15
       group B              65.93          62.85          60.22
       group C              67.61          65.42          62.71
       group D              69.41          66.14          65.41
       group E              76.75          70.30          67.39

Average Scores by Parental Education x Gender:
                                    math score  reading score  writing score
parental level of education gende

In [16]:
# --- 4. UFUNC TRANSFORMATIONS ---
print('-- 4. UFUNC TRANSFORMATIONS (API Summary) --')
weights = np.array([0.40, 0.30, 0.30])
print(f'Weights: Math={weights[0]}, Reading={weights[1]}, Writing={weights[2]}')
print(f'API range : {df["API"].min():.2f} - {df["API"].max():.2f}')
print(f'API mean  : {df["API"].mean():.2f}')
print(f'API std   : {df["API"].std():.2f}')

# API by education level
api_by_edu = df.groupby('parental level of education')['API'].mean().sort_values(ascending=False)
print('\nAverage API by Parental Education Level:')
print(api_by_edu.round(2))

# API by gender
api_by_gender = df.groupby('gender')['API'].agg(['mean','std','min','max']).round(2)
print('\nAPI Statistics by Gender:')
print(api_by_gender)

# ufunc: normalize scores to 0-1 range
score_cols = ['math score', 'reading score', 'writing score']
score_min = df[score_cols].values.min()
score_max = df[score_cols].values.max()
normalized = np.divide(np.subtract(df[score_cols].values, score_min), (score_max - score_min))
df_norm = pd.DataFrame(normalized, columns=['math_norm', 'reading_norm', 'writing_norm'])
print('\nNormalized Score Stats (ufunc: subtract + divide):')
print(df_norm.describe().round(4))

-- 4. UFUNC TRANSFORMATIONS (API Summary) --
Weights: Math=0.4, Reading=0.3, Writing=0.3
API range : 8.10 - 100.00
API mean  : 67.60
API std   : 14.24

Average API by Parental Education Level:
parental level of education
master's degree       73.21
bachelor's degree     71.67
associate's degree    69.40
some college          68.34
some high school      64.95
high school           63.00
Name: API, dtype: float64

API Statistics by Gender:
         mean    std   min    max
gender                           
female  68.98  14.59   8.1  100.0
male    66.13  13.71  23.7  100.0

Normalized Score Stats (ufunc: subtract + divide):
       math_norm  reading_norm  writing_norm
count  1000.0000     1000.0000     1000.0000
mean      0.6609        0.6917        0.6805
std       0.1516        0.1460        0.1520
min       0.0000        0.1700        0.1000
25%       0.5700        0.5900        0.5775
50%       0.6600        0.7000        0.6900
75%       0.7700        0.7900        0.7900
max       

In [18]:
# --- 5. GRADE DISTRIBUTION ---
print('-- 5. GRADE DISTRIBUTION --')
grade_cols = ['math grade', 'reading grade', 'writing grade']
for col in grade_cols:
    print(f'\n{col}:')
    counts = df[col].value_counts().sort_index()
    pcts   = (counts / len(df) * 100).round(2)
    grade_summary = pd.DataFrame({'Count': counts, 'Percentage': pcts})
    print(grade_summary)

# Grade distribution by test preparation course
print('\nMath Grade by Test Preparation Course:')
print(pd.crosstab(df['test preparation course'], df['math grade']))

print('\nWriting Grade by Test Preparation Course:')
print(pd.crosstab(df['test preparation course'], df['writing grade']))

# --- 6. TOP / BOTTOM SUMMARY ---
print('\n-- 6. TOP & BOTTOM PERFORMERS SUMMARY --')
top5 = df.nlargest(5, 'API')[['gender','race/ethnicity','parental level of education',
                               'math score','reading score','writing score','API','Rank']]
print('Top 5:')
print(top5.to_string(index=False))

bot5 = df.nsmallest(5, 'API')[['gender','race/ethnicity','parental level of education',
                                'math score','reading score','writing score','API','Rank']]
print('\nBottom 5:')
print(bot5.to_string(index=False))



-- 5. GRADE DISTRIBUTION --

math grade:
            Count  Percentage
math grade                   
A              58         5.8
B             135        13.5
C             216        21.6
D             268        26.8
F             323        32.3

reading grade:
               Count  Percentage
reading grade                   
A                 79         7.9
B                170        17.0
C                264        26.4
D                233        23.3
F                254        25.4

writing grade:
               Count  Percentage
writing grade                   
A                 78         7.8
B                157        15.7
C                254        25.4
D                230        23.0
F                281        28.1

Math Grade by Test Preparation Course:
math grade                A   B    C    D    F
test preparation course                       
completed                32  56   88   95   87
none                     26  79  128  173  236

Writing Grade by Test Prep